# Mitwork Procurement Anomaly Detection
### Reproducible retrospective screening study | Revised v3 — specification-extension, conditional retrospective

**Research design.** We analyze four Mitwork exports and compare exactly three approaches: (1) a five-indicator red-flag baseline, (2) tabular Isolation Forest, and (3) graph-enriched Isolation Forest.

**Period:** October 2025 through September 2026. **Split:** train October–June, validation July, test August–September. All eligible 115,177 lots are retained; the earlier 100,000-lot cap has been explicitly waived for this study.

> **Important scientific limitation:** The current exports do **not** prove when bidder identities, current field values, or historical relationships were first publicly accessible. Therefore this is a **retrospective anomaly-screening experiment**, **not** certified prospective fraud detection. No model score is a proven indicator of corruption. No Precision@K is computed without independent reviewer labels.

The notebook produces reproducible CSV reports and PDF charts outside `data/raw`. Raw records, lot identifiers and review worksheets must **not** be published in a public repository.

# Part I — Source audit and descriptive analytics
The original audit is retained as the starting point. All outputs below will be re-executed because subsequent features and models have changed.

## 0. Environment and imports
Run cells from top to bottom in JupyterLab. If packages are missing, run `%pip install pandas pyarrow numpy matplotlib seaborn` in a cell, restart the kernel, and rerun.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams.update({"figure.figsize": (11, 5), "axes.titlesize": 14})

# Works if the notebook is saved in Downloads OR Downloads/notebooks.
CANDIDATES = [Path.cwd() / "data" / "raw", Path.cwd().parent / "data" / "raw",
              Path.home() / "Downloads" / "data" / "raw"]
DATA_DIR = next((p for p in CANDIDATES if (p / "mw_lots.parquet").exists()), None)
if DATA_DIR is None:
    raise FileNotFoundError("Could not find data/raw. Edit CANDIDATES to your local data directory.")
ROOT = DATA_DIR.parent.parent
OUT = ROOT / "outputs" / "01_audit"
OUT.mkdir(parents=True, exist_ok=True)
print("Data:", DATA_DIR)
print("Output:", OUT)
print("pandas:", pd.__version__)

## 1. Manifest and source integrity
Read the export manifest and compare recorded and loaded row counts. SHA-256 hashes are retained as provenance; hashing large files is optional.

In [ ]:
manifest_path = DATA_DIR / "export_manifest.json"
manifest = json.loads(manifest_path.read_text(encoding="utf-8")) if manifest_path.exists() else {}
print("Exported at (UTC):", manifest.get("exported_at_utc", "not available"))
print("Database:", manifest.get("database", "not available"))
print("Raw tables are not filtered or deduplicated.")

## 2. Load source tables

In [ ]:
names = ["mw_buys", "mw_lots", "mw_offers", "mw_subjects"]
tables = {name: pd.read_parquet(DATA_DIR / f"{name}.parquet") for name in names}
buys, lots, offers, subjects = (tables[name] for name in names)
summary = pd.DataFrame([
    {"table": name, "rows": len(df), "columns": len(df.columns),
     "manifest_rows": manifest.get("tables", {}).get(name, {}).get("rows")}
    for name, df in tables.items()
])
summary["count_matches"] = summary["rows"].eq(summary["manifest_rows"])
display(summary)
summary.to_csv(OUT / "data_account.csv", index=False)


## 3. Column inventory and data types

In [ ]:
inventory = pd.concat([
    pd.DataFrame({"table": name, "column": df.columns.astype(str),
                  "dtype": df.dtypes.astype(str).values})
    for name, df in tables.items()
], ignore_index=True)
display(inventory.groupby("table").size().rename("number_of_columns"))
display(inventory)
inventory.to_csv(OUT / "column_inventory.csv", index=False)

## 4. Missing values
For readability, the figure shows the twenty columns with the highest missing-value fractions within each table.

In [ ]:
missing = pd.concat([
    pd.DataFrame({"table": name, "column": df.columns.astype(str),
                  "missing_count": df.isna().sum().values,
                  "missing_pct": (100 * df.isna().mean()).values})
    for name, df in tables.items()
], ignore_index=True)
missing = missing.sort_values(["table", "missing_pct"], ascending=[True, False])
display(missing.groupby("table").head(12).round(2))
missing.to_csv(OUT / "missingness.csv", index=False)
fig, axs = plt.subplots(2, 2, figsize=(15, 11), constrained_layout=True)
for ax, name in zip(axs.flat, names):
    frame = missing[missing.table == name].head(15).sort_values("missing_pct")
    ax.barh(frame["column"], frame["missing_pct"])
    ax.set_title(name)
    ax.set_xlabel("Missing (%)")
    ax.set_xlim(0, 100)
fig.suptitle("Missingness by Source Table", fontsize=17)
fig.savefig(OUT / "missingness_by_table.png", dpi=180, bbox_inches="tight")
plt.show()

## 5. Primary-key checks

In [ ]:
id_checks = []
for name, df in tables.items():
    if "id" in df.columns:
        id_checks.append({"table": name, "rows": len(df),
                          "null_id": int(df["id"].isna().sum()),
                          "duplicate_id": int(df["id"].duplicated().sum()),
                          "unique_id": int(df["id"].nunique(dropna=True))})
id_checks = pd.DataFrame(id_checks)
display(id_checks)
id_checks.to_csv(OUT / "id_checks.csv", index=False)

## 6. Foreign-key and relationship validation
Validate `mw_lots.trd_buys_id → mw_buys.id` and `mw_offers.trd_lots_id → mw_lots.id` without assuming joins are perfect.

In [ ]:
relation_checks = []
for source, parent, source_fk, parent_pk in [
    ("mw_lots", "mw_buys", "trd_buys_id", "id"),
    ("mw_offers", "mw_lots", "trd_lots_id", "id")]:
    child, master = tables[source], tables[parent]
    if source_fk in child and parent_pk in master:
        orphan = child[source_fk].notna() & ~child[source_fk].isin(master[parent_pk])
        relation_checks.append({"relation": f"{source}.{source_fk} → {parent}.{parent_pk}",
                                "null_foreign_key": int(child[source_fk].isna().sum()),
                                "orphan_nonnull_rows": int(orphan.sum()),
                                "child_rows": len(child)})
    else:
        print("Columns unavailable for:", source, parent)
relations = pd.DataFrame(relation_checks)
display(relations)
relations.to_csv(OUT / "relationship_checks.csv", index=False)

## 7. Competition: actual offers per lot
Zero offers in the extracted table do not necessarily prove that no bids were submitted on the portal. Treat zero as **no observed offer rows**.

In [ ]:
if "trd_lots_id" in offers.columns and "id" in lots.columns:
    observed = offers.groupby("trd_lots_id").size().rename("observed_offers")
    lot_offers = lots[["id"] + (["offers_count"] if "offers_count" in lots.columns else [])].copy()
    lot_offers = lot_offers.join(observed, on="id")
    lot_offers["observed_offers"] = lot_offers["observed_offers"].fillna(0).astype(int)
    lot_offers["category"] = pd.cut(lot_offers["observed_offers"],
        bins=[-1, 0, 1, np.inf], labels=["0 observed", "1 observed", "2+ observed"])
    category_counts = lot_offers["category"].value_counts(sort=False)
    display(category_counts.to_frame("lots"))
    fig, ax = plt.subplots()
    ax.bar(category_counts.index.astype(str), category_counts.values)
    ax.set(title="Observed Offer Rows per Lot", ylabel="Number of lots", xlabel="")
    fig.savefig(OUT / "competition_observed.png", dpi=180, bbox_inches="tight")
    plt.show()
    if "offers_count" in lot_offers:
        declared = pd.to_numeric(lot_offers["offers_count"], errors="coerce")
        comparison = pd.DataFrame({"declared_null": [int(declared.isna().sum())],
            "non_null_mismatch": [int((declared.notna() & declared.ne(lot_offers["observed_offers"])).sum())]})
        display(comparison)
        comparison.to_csv(OUT / "offer_count_comparison.csv", index=False)
else:
    print("Offer-to-lot linkage columns not available.")

## 8. Temporal data availability and pilot window
The bid submission deadline (`mw_buys.end_date`) is a candidate chronological anchor, **not** verified bid-opening time. Time-based split: nine train months, one validation month, two test months; no random subsampling.

In [ ]:
date_inventory = []
for name, df in tables.items():
    for col in df.columns:
        if "date" in col.lower() or "time" in col.lower() or col.lower().endswith("_at"):
            dates = pd.to_datetime(df[col], errors="coerce", utc=True)
            date_inventory.append({"table": name, "column": col,
                "parseable_rows": int(dates.notna().sum()),
                "min_utc": str(dates.min()), "max_utc": str(dates.max())})
dates_report = pd.DataFrame(date_inventory)
display(dates_report)
dates_report.to_csv(OUT / "temporal_columns.csv", index=False)

In [ ]:
if {"id", "end_date"}.issubset(buys.columns) and "trd_buys_id" in lots.columns:
    ref = buys[["id", "end_date"]].drop_duplicates(subset="id").copy()
    ref["end_date"] = pd.to_datetime(ref["end_date"], errors="coerce", utc=True)
    by_lot = lots[["id", "trd_buys_id"]].merge(ref, left_on="trd_buys_id", right_on="id", how="left", suffixes=("_lot", "_buy"))
    by_lot["month"] = by_lot["end_date"].dt.to_period("M")
    monthly = by_lot.groupby("month", dropna=True).size().rename("lots").reset_index()
    monthly["month"] = monthly["month"].astype(str)
    display(monthly.tail(24))
    monthly.to_csv(OUT / "monthly_coverage.csv", index=False)
    fig, ax = plt.subplots(figsize=(14, 5))
    recent = monthly.tail(30)
    ax.bar(recent["month"], recent["lots"])
    ax.tick_params(axis="x", rotation=65)
    ax.set(title="Lots by Bid Deadline Month", xlabel="Deadline month (UTC)", ylabel="Lots")
    fig.tight_layout()
    fig.savefig(OUT / "monthly_lots.png", dpi=180, bbox_inches="tight")
    plt.show()
    # Provisional pre-agreed Oct 2025 - Sep 2026 period; all counts are pre-cleaning.
    pilot = monthly[(monthly.month >= "2025-10") & (monthly.month <= "2026-09")].copy()
    pilot["split"] = np.select([pilot.month <= "2026-06", pilot.month == "2026-07"], ["train", "validation"], default="test")
    print("FULL 12-MONTH STUDY: Oct 2025 - Sep 2026; lot cap explicitly waived")
    display(pilot.groupby("split", sort=False)["lots"].sum().to_frame("lots"))
    display(pilot)
    pilot.to_csv(OUT / "provisional_pilot_months.csv", index=False)
else:
    print("Required columns missing: cannot generate monthly analysis automatically.")

## 9. Leakage-risk checklist
Before feature construction, verify every source timestamp relative to the prediction cutoff. The supplied tables do not yet establish a reliable bid-opening timestamp. The following features must **not** be computed for the current lot until their availability at prediction time is proven:

- Current-lot participants or offer prices
- Winning supplier, winner flag, final awarded price
- Post-deadline revisions, cancellation, outcomes
- Edges in the graph constructed from the current lot’s future bids

**Graph lookback:** 180 days preceding each prediction cutoff. Historical edges must be point-in-time valid. A missing bid-opening timestamp is a substantive research limitation, not a reason to relax the leakage restriction.

## 10. Research scope and cutoff caveats

* Full 12-month population is retained (no 100,000-lot truncation).
* `mw_buys.end_date` defines the chronology; it does **not** establish a verified public bid-opening moment.
* Source exports are snapshots. The historical versions of lot amounts, methods, categories and publication dates are not available.
* Historical offers and buyer–supplier graphs may include participants who were not publicly visible until after the prediction date. This must be reported as an unverified availability restriction.
* Reissued / parent-linked lots can cross chronological splits. We perform a *component-disjoint sensitivity analysis* later rather than silently throwing away the original observations.

In [ ]:
print("Audit complete. Reports and figures:", OUT)
print("Generated files:")
for p in sorted(OUT.iterdir()):
    print(" -", p.name)

---
# Part II — Pilot selection and source quality
**Study decision:** keep the full 12-month window without random sampling. The 100,000-lot administrative cap is waived. The offer cap (1 million) is not binding.

In [ ]:
PILOT_START = pd.Timestamp('2025-10-01', tz='UTC')
PILOT_END_EXCLUSIVE = pd.Timestamp('2026-10-01', tz='UTC')
LOT_CAP = None  # Waived for this research iteration
OFFER_CAP = 1_000_000
REPORT_DIR = ROOT / 'outputs' / 'master_notebook'
PROCESSED_DIR = ROOT / 'data' / 'processed'
REPORT_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
print('Study window:', PILOT_START, 'through', PILOT_END_EXCLUSIVE, '(exclusive)')
print('Study design: full 12-month cohort, retrospective screening')

## 2.1 Identify null keys, duplicate IDs and unusable deadlines

The **cutoff proxy** is `mw_buys.end_date` only; it does **not** prove that sealed participant identities or bid prices were visible. We keep IDs as strings in research exports and convert exact numeric strings only when needed for plotting or future feature calculations.

In [ ]:
# Merge only the minimal required columns; avoid copying 400 MB of lot descriptions.
required_lot_cols = ["id", "trd_buys_id", "parent_id", "sys_subjects_id", "amount", "ref_enstru_code"]
required_buy_cols = ["id", "end_date", "start_date", "create_date", "sys_subjects_id", "ref_trade_method_code"]
for col in required_lot_cols:
    assert col in lots.columns, f"Missing mw_lots.{col}"
for col in required_buy_cols:
    assert col in buys.columns, f"Missing mw_buys.{col}"

lot_base = lots[required_lot_cols].copy()
buy_lookup = buys[required_buy_cols].rename(columns={
    "id": "buy_key", "sys_subjects_id": "buy_buyer_key"
})
cohort = lot_base.merge(buy_lookup, left_on="trd_buys_id", right_on="buy_key",
                        how="left", validate="many_to_one")
cohort["cutoff_utc"] = pd.to_datetime(cohort["end_date"], errors="coerce", utc=True)
cohort["start_utc"] = pd.to_datetime(cohort["start_date"], errors="coerce", utc=True)
cohort["lot_id"] = cohort["id"].astype("string")
cohort["buy_id"] = cohort["trd_buys_id"].astype("string")
# Buyer: lot field preferred when available; buy organizer is only a fallback proxy.
cohort["buyer_id"] = cohort["sys_subjects_id"].fillna(cohort["buy_buyer_key"]).astype("string")
cohort["amount_kzt"] = pd.to_numeric(cohort["amount"], errors="coerce")

invalid_id = cohort["lot_id"].isna()
duplicate_id = cohort["lot_id"].duplicated(keep="first") & ~invalid_id
invalid_cutoff = cohort["cutoff_utc"].isna()
invalid_buy = cohort["buy_key"].isna()
negative_amount = cohort["amount_kzt"].lt(0)

# Sequential, disjoint accounting. No imputation of missing identifiers or deadlines.
reasons = np.select(
    [invalid_id, duplicate_id, invalid_cutoff],
    ["missing_lot_id", "duplicate_lot_id", "missing_deadline"],
    default="eligible_for_version_review"
)
cohort["audit_reason"] = reasons
stage_counts = cohort["audit_reason"].value_counts().rename_axis("reason").reset_index(name="lots")
display(stage_counts)
print("Broken buy joins:", int(invalid_buy.sum()))
print("Negative lot amounts (AUDIT ONLY, not dropped):", int(negative_amount.sum()))
print("Rows before:", len(cohort), "after basic eligibility:", int((cohort.audit_reason == 'eligible_for_version_review').sum()))
stage_counts.to_csv(REPORT_DIR / "02_exclusion_account.csv", index=False)


## 2.2 Historical versions and possible split-boundary conflicts

`mw_lots.parent_id` may refer to a previous version, while `mw_lots.id` is unique. A parent link alone does not establish that both rows should be merged or discarded. We flag **direct parent-child links crossing temporal splits** for explicit review; we do not silently drop them.

In [ ]:
eligible_all = cohort.loc[cohort["audit_reason"].eq("eligible_for_version_review")].copy()
pilot_mask = (eligible_all.cutoff_utc >= PILOT_START) & (eligible_all.cutoff_utc < PILOT_END_EXCLUSIVE)
pilot = eligible_all.loc[pilot_mask].copy()
pilot["month"] = pilot.cutoff_utc.dt.strftime("%Y-%m")
pilot["split"] = np.select(
    [pilot["month"] <= "2026-06", pilot["month"].eq("2026-07")],
    ["train", "validation"], default="test"
)
# Monotonic/split assertions before any modeling.
assert pilot["lot_id"].notna().all()
assert pilot["lot_id"].is_unique
assert (pilot["cutoff_utc"] >= PILOT_START).all()
assert (pilot["cutoff_utc"] < PILOT_END_EXCLUSIVE).all()
assert set(pilot["split"]).issubset({"train", "validation", "test"})

# Avoid floating parent IDs and potential 64-bit precision loss in research exports.
# PostgreSQL BIGINT values are safely representable here when <2**53, verified below.
parent_vals = pd.to_numeric(pilot["parent_id"], errors="coerce")
max_parent = parent_vals.abs().max()
if pd.notna(max_parent):
    assert max_parent < 2**53, "Potential float parent-ID precision loss: review source representation"
pilot["parent_lot_id"] = parent_vals.astype("Int64").astype("string")
parent_links = pilot[["lot_id", "parent_lot_id", "split"]].merge(
    pilot[["lot_id", "split"]].rename(columns={"lot_id": "parent_lot_id", "split": "parent_split"}),
    on="parent_lot_id", how="inner", validate="many_to_one"
)
parent_links = parent_links.loc[parent_links["lot_id"].ne(parent_links["parent_lot_id"])].copy()
cross_split_links = parent_links[parent_links["split"].ne(parent_links["parent_split"])].copy()
print("Direct version links within pilot:", len(parent_links))
print("Direct version links crossing split boundaries:", len(cross_split_links))
if len(cross_split_links):
    display(cross_split_links.head(10))
cross_split_links.to_csv(REPORT_DIR / "02_cross_split_version_links.csv", index=False)


### 2.2a Connected components across the **entire** lot-version graph
Direct parent-child checks can miss test lots linked to training lots through longer chains or through a shared ancestor outside the pilot. Build components over all exported `mw_lots`, then identify *whole components* appearing in more than one split. This does not change the primary sample.

In [ ]:
# Union-find over all 674k exported lots; parent IDs are float in the export, so verify exactness.
source_ids=pd.to_numeric(lots['id'],errors='raise').astype('int64').to_numpy()
source_parents=pd.to_numeric(lots['parent_id'],errors='coerce')
if source_parents.notna().any():
    assert source_parents.dropna().abs().max()<2**53, 'Parent ID precision must be audited'
uf={int(i):int(i) for i in source_ids}
def uf_find(x):
    while uf[x]!=x:
        uf[x]=uf[uf[x]]
        x=uf[x]
    return x
def uf_join(a,b):
    ra,rb=uf_find(a),uf_find(b)
    if ra!=rb:uf[max(ra,rb)]=min(ra,rb)
for child,parent in zip(source_ids,source_parents.to_numpy()):
    if pd.notna(parent) and int(parent) in uf and int(child)!=int(parent):
        uf_join(int(child),int(parent))
pilot['version_component_id']=pilot['id'].map(lambda x:uf_find(int(x))).astype('int64')
component_table=pilot.groupby('version_component_id',observed=True).agg(
    lots=('lot_id','size'),n_splits=('split','nunique'),first_cutoff=('cutoff_utc','min'),
    last_cutoff=('cutoff_utc','max')).reset_index()
cross_components=set(component_table.loc[component_table['n_splits']>1,'version_component_id'])
pilot['cross_split_component']=pilot['version_component_id'].isin(cross_components)
print('Version components represented in pilot:',len(component_table))
print('Cross-split components:',len(cross_components))
print('Lots belonging to cross-split components:',int(pilot['cross_split_component'].sum()))
display(pilot.groupby(['split','cross_split_component']).size().to_frame('lots'))
component_table.to_csv(REPORT_DIR/'02_version_components_RESTRICTED.csv',index=False)
assert pilot.loc[~pilot['cross_split_component']].groupby('version_component_id')['split'].nunique().le(1).all()

## 2.3 Pilot population and coverage decision

The chronological cut is fixed at 9/1/2 calendar months. Cross-boundary version links mean **not ready for publication-grade modeling**. For research reproducibility, this notebook nevertheless exports a clearly labeled *candidate* dataset for further diagnostics.

In [ ]:
# Exhaustive 12-month pilot: administrative lot cap explicitly waived.
pilot_offer_rows = int(offers['trd_lots_id'].isin(pilot['id']).sum())
month_table = pilot.groupby(['month','split'], observed=True).agg(
    lots=('lot_id','size'), min_cutoff=('cutoff_utc','min'), max_cutoff=('cutoff_utc','max')
).reset_index()
cap_status = pd.DataFrame([
    {'constraint':'lot cap (waived)','observed':len(pilot),'allowed':pd.NA,'status':'waived'},
    {'constraint':'offer cap','observed':pilot_offer_rows,'allowed':OFFER_CAP,
     'status':'pass' if pilot_offer_rows <= OFFER_CAP else 'exceeded'}
])
display(cap_status)
display(month_table)
display(pilot.groupby('split').size().to_frame('lots'))
assert pilot_offer_rows <= OFFER_CAP, 'Offer cap exceeded; document another study-design change.'
assert len(pilot) == 115177, 'Population changed relative to verified export — investigate before interpretation.'
cap_status.to_csv(REPORT_DIR/'02_cap_audit.csv',index=False)
month_table.to_csv(REPORT_DIR/'02_monthly_pilot.csv',index=False)
fig,ax=plt.subplots(figsize=(13,4.5))
color_map={'train':'#4c78a8','validation':'#f2a541','test':'#54a24b'}
for split_name,grp in month_table.groupby('split',sort=False):
    ax.bar(grp['month'],grp['lots'],label=split_name,color=color_map[split_name])
ax.legend(title='Chronological split',frameon=False)
ax.set(title='Lots by bidding deadline month (UTC)',xlabel='Month',ylabel='Lot count')
ax.tick_params(axis='x',rotation=45)
fig.tight_layout();fig.savefig(REPORT_DIR/'02_pilot_monthly_lots.pdf',bbox_inches='tight');plt.show()

In [ ]:
keep=['lot_id','buy_id','buyer_id','parent_lot_id','cutoff_utc','start_utc',
      'month','split','amount_kzt','ref_enstru_code','ref_trade_method_code','audit_reason']
pilot[keep].to_parquet(PROCESSED_DIR/'pilot_full_12_months_RESTRICTED.parquet',index=False)
print('Retrospective full-period candidate saved; point-in-time safety NOT certified.')

---
# Part III — Cutoff Semantics & Leakage Feasibility

**Critical finding.** Available timestamp fields include `end_date` and `apply_date` but **no independently confirmed bid-opening/revelation timestamp**. Thus, the default prediction cutoff is `mw_buys.end_date` and **participant lists, current bid prices, offer counts and winner outcomes are ineligible** until a disclosure timestamp is verified. Historical edges also require evidence that their contents were observable before each prediction.

The table below is a methodological gate, not a claim that unavailable variables are zero.

In [ ]:
leakage_gate=pd.DataFrame([
 {'feature_family':'Lot amount, category and trade method','status':'snapshot-only / not certified','reason':'No field-level immutable publication history'},
 {'feature_family':'Bidding window (start/end)','status':'snapshot-only / not certified','reason':'Amendments may have changed deadlines'},
 {'feature_family':'Prior buyer lot deadlines','status':'retrospective / strict earlier timestamps','reason':'Cannot verify whether previous records were publicly known'},
 {'feature_family':'Current lot offer counts, identities or prices','status':'EXCLUDED','reason':'No verified bid opening/public disclosure timestamp'},
 {'feature_family':'Winner and award outcomes','status':'EXCLUDED','reason':'Outcome information arrives after prediction cutoff'},
 {'feature_family':'Prior buyer-supplier graph','status':'RETROSPECTIVE ONLY','reason':'30-day availability buffer does not prove disclosure'},
 {'feature_family':'Parent/child and reissue relationships','status':'AUDITED / not predictive feature','reason':'Possible shared latent event across splits'}
])
display(leakage_gate)
leakage_gate.to_csv(REPORT_DIR/'03_feature_eligibility.csv',index=False)

## 3.1 Automated boundary checks

These assertions verify that every candidate's own deadline lies in the expected chronological split, but **cannot certify historical information visibility**. Such visibility requires source documentation or a verified timestamp.

In [ ]:
windows = {
    "train": ("2025-10-01", "2026-07-01"),
    "validation": ("2026-07-01", "2026-08-01"),
    "test": ("2026-08-01", "2026-10-01"),
}
for split_name, (lo, hi) in windows.items():
    part = pilot.loc[pilot["split"].eq(split_name)]
    assert part.cutoff_utc.ge(pd.Timestamp(lo, tz="UTC")).all(), split_name
    assert part.cutoff_utc.lt(pd.Timestamp(hi, tz="UTC")).all(), split_name
assert pilot["lot_id"].is_unique, "A lot cannot appear in two splits"
print("PASS: chronological windows and unique lot IDs")
print("WARNING: this is NOT verification of bid-opening visibility or prior-event public disclosure.")


### 3.1a Snapshot provenance and post-cutoff timestamp audit
An extracted record created or updated after the candidate deadline is **not** a verified point-in-time feature snapshot. Report counts; do not automatically claim that creation/update date is the public disclosure date.

In [ ]:
# Audit timestamp fields on lots and buys; do not treat database timestamps as legal publication proof.
check=pilot[['id','trd_buys_id','cutoff_utc','split']].copy()
lot_time=lots[['id','create_date','update_date']].rename(columns={'create_date':'lot_created','update_date':'lot_updated'})
buy_time=buys[['id','create_date','update_date']].rename(columns={
    'id':'buy_lookup_id','create_date':'buy_created','update_date':'buy_updated'})
check=check.merge(lot_time,on='id',how='left',validate='one_to_one')
check=check.merge(buy_time,left_on='trd_buys_id',right_on='buy_lookup_id',how='left',validate='many_to_one')
provenance_rows=[]
for col in ['lot_created','lot_updated','buy_created','buy_updated']:
    t=pd.to_datetime(check[col],errors='coerce',utc=True)
    after=t>check['cutoff_utc']
    provenance_rows.append({'timestamp_field':col,'missing_share':float(t.isna().mean()),
                            'after_deadline_count':int(after.sum()),'after_deadline_share':float(after.mean())})
provenance_audit=pd.DataFrame(provenance_rows)
display(provenance_audit.round(4))
provenance_audit.to_csv(REPORT_DIR/'03_snapshot_timestamp_audit.csv',index=False)
print('TIMESTAMP WARNING: updating a record after the cutoff is evidence of snapshot risk, not proof of which fields changed.')
del check,lot_time,buy_time

## 3.2 What the leakage checks do *and do not* establish

We enforce strictly earlier source-deadline events for historical aggregates, exclude current-lot offers and outcomes, and audit version-component crossover. We also document after-deadline creation/update timestamps.

**These programmatic safeguards do not turn a retrospective snapshot into a point-in-time archive.** A prospective claim would require dated publication records, procurement amendments, and confirmation that historical bidders were observable before each cutoff.

---
# Part IV — Feature engineering with strict event-time ordering

We construct numerical and categorical predictors using no current-lot offer data and no award outcomes. Same-time lot groups receive the **same prior counts**, rather than using an arbitrary internal row order. Historical counts extend before October 2025 when data permit.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import IsolationForest
from sklearn.base import clone
from scipy.stats import spearmanr
import warnings, json, gc, sys

MODEL_DIR=ROOT/'outputs'/'full_research_v2'
MODEL_DIR.mkdir(parents=True,exist_ok=True)
RANDOM_STATE=42
SEEDS=[11,21,42,84,101]
LOOKBACK_DAYS=180
EMBARGO_DAYS=30
print('Study population:',len(pilot),'lots; observed source offers:',len(offers))
assert pilot['lot_id'].is_unique and pilot['cutoff_utc'].notna().all()

In [ ]:
# Make a compact feature frame from the previously audited time-window population.
f=pilot[['lot_id','buy_id','buyer_id','parent_lot_id','cutoff_utc','start_utc','split','month',
         'amount_kzt','ref_enstru_code','ref_trade_method_code']].copy()
f['duration_days']=(f['cutoff_utc']-f['start_utc']).dt.total_seconds()/86400
f['log_amount']=np.log1p(f['amount_kzt'].clip(lower=0))
f['category2']=f['ref_enstru_code'].astype('string').str[:2].fillna('UNKNOWN')
f['trade_method']=f['ref_trade_method_code'].astype('string').fillna('UNKNOWN')
f['month_number']=f['cutoff_utc'].dt.month.astype('int16')
f['weekday']=f['cutoff_utc'].dt.dayofweek.astype('int8')
f['duration_bad']=f['duration_days'].lt(0).fillna(False)
f=f.sort_values(['cutoff_utc','lot_id']).reset_index(drop=True)

# Do NOT use groupby(...).cumcount() on individually ordered rows: it leaks same-timestamp lots.
# Use all historical lots to compute event counts strictly BEFORE the candidate lot cutoff.
historical_lots=cohort[['id','buyer_id','cutoff_utc','ref_enstru_code']].dropna(
    subset=['id','buyer_id','cutoff_utc']).drop_duplicates('id').copy()
historical_lots=historical_lots.loc[historical_lots['cutoff_utc']<PILOT_END_EXCLUSIVE]
historical_lots['category2']=historical_lots['ref_enstru_code'].astype('string').str[:2].fillna('UNKNOWN')
historical_lots['buyer_id']=historical_lots['buyer_id'].astype('string')

def strictly_prior_count(source, keys, out_name):
    # Counts by timestamp, excluding observations with equal timestamps.
    by=source.groupby(keys+['cutoff_utc'],dropna=True,observed=True).size().rename('at_time').reset_index()
    by=by.sort_values(keys+['cutoff_utc']).reset_index(drop=True)
    by[out_name]=by.groupby(keys,dropna=True,observed=True)['at_time'].cumsum()-by['at_time']
    if (by[out_name]<0).any(): raise AssertionError('Negative prior count')
    return by[keys+['cutoff_utc',out_name]]

bt=strictly_prior_count(historical_lots,['buyer_id'],'buyer_prior_lots')
bct=strictly_prior_count(historical_lots,['buyer_id','category2'],'buyer_category_prior_lots')
f=f.merge(bt,on=['buyer_id','cutoff_utc'],how='left',validate='many_to_one')
f=f.merge(bct,on=['buyer_id','category2','cutoff_utc'],how='left',validate='many_to_one')
for c in ['buyer_prior_lots','buyer_category_prior_lots']:
    f[c]=f[c].fillna(0).astype('float32')
assert len(f)==len(pilot) and f['lot_id'].is_unique
# Sanity test: all lots with the same buyer and time must receive identical prior totals.
assert f.groupby(['buyer_id','cutoff_utc'],dropna=True)['buyer_prior_lots'].nunique().le(1).all()
print('Prior-event counts use strictly earlier timestamps, with ties treated simultaneously.')
display(f[['duration_days','amount_kzt','buyer_prior_lots','buyer_category_prior_lots']].isna().mean().to_frame('missing_share'))

## Five exploratory baseline red flags (train-only calibration)

The five indicators are **statistical proxy heuristics**, not established legal red flags. We use extreme *training-period* quantiles rather than automatically flagging half of all lots as having a short window. Unseen trade methods are not automatically assumed risky. Do not interpret a large flag count as evidence of misconduct.

In [ ]:
train=f['split'].eq('train').to_numpy();val=f['split'].eq('validation').to_numpy();test=f['split'].eq('test').to_numpy()
assert train.sum()>0 and val.sum()>0 and test.sum()>0
valid_dur=f.loc[train & f['duration_days'].ge(0).to_numpy(),['trade_method','duration_days']]
method_duration=valid_dur.groupby('trade_method')['duration_days'].agg(['size',lambda x:x.quantile(.10)])
method_duration.columns=['n','q10']
global_short=valid_dur['duration_days'].quantile(.10)
short_cut=method_duration['q10'].where(method_duration['n']>=30).to_dict()
short_threshold=f['trade_method'].map(short_cut).fillna(global_short)
amount_p95=f.loc[train,'amount_kzt'].quantile(.95)
buyer_p95=f.loc[train,'buyer_prior_lots'].quantile(.95)
category_p95=f.loc[train,'buyer_category_prior_lots'].quantile(.95)
method_freq=f.loc[train,'trade_method'].value_counts(normalize=True)
f['flag_short_window']=(f['duration_days'].ge(0) & f['duration_days'].lt(short_threshold)).astype('int8')
f['flag_large_amount']=f['amount_kzt'].gt(amount_p95).fillna(False).astype('int8')
f['flag_repeat_buyer']=f['buyer_prior_lots'].gt(buyer_p95).astype('int8')
f['flag_repeat_category']=f['buyer_category_prior_lots'].gt(category_p95).astype('int8')
f['flag_rare_method']=(f['trade_method'].isin(method_freq.index) &
                       f['trade_method'].map(method_freq).lt(.01).fillna(False)).astype('int8')
flag_cols=['flag_short_window','flag_large_amount','flag_repeat_buyer','flag_repeat_category','flag_rare_method']
f['red_flag_score']=f[flag_cols].sum(axis=1).astype('int8')
flag_prevalence=f.groupby('split')[flag_cols].mean().T
print('Train quantiles:',{'amount_95':float(amount_p95),'buyer_history_95':float(buyer_p95),
                         'buyer_category_history_95':float(category_p95),'duration_10':float(global_short)})
display(flag_prevalence.round(3));display(f.groupby('split')['red_flag_score'].describe())
flag_prevalence.to_csv(MODEL_DIR/'04_baseline_flag_prevalence.csv')

# Part V — Red-flag baseline

The baseline is the sum of the five proxy binary indicators above. All quantiles and frequencies are estimated on the 9-month training window. There is no target label; percentages marked Top K represent **review budgets**.

In [ ]:
plt.figure(figsize=(9,4))
sns.countplot(data=f.loc[test],x='red_flag_score',color='#5B8DB8')
plt.title('Five-flag baseline — test-period score distribution')
plt.xlabel('Number of triggered flags');plt.ylabel('Lots');plt.tight_layout()
plt.savefig(MODEL_DIR/'05_flag_distribution.pdf',bbox_inches='tight');plt.show()

# Part VI — Tabular Isolation Forest
Tabular IF is trained **only** on October 2025–June 2026. No confirmed anomaly labels are available, so validation reports unsupervised score stability, not fraud-detection accuracy. The test set is never used for hyperparameter selection.

In [ ]:
num_cols=['log_amount','duration_days','buyer_prior_lots','buyer_category_prior_lots','month_number','weekday']
cat_cols=['trade_method','category2']
X_cols=num_cols+cat_cols
num_pipe=Pipeline([('impute',SimpleImputer(strategy='median')),('scale',StandardScaler())])
cat_pipe=Pipeline([('impute',SimpleImputer(strategy='most_frequent')),
                   ('onehot',OneHotEncoder(handle_unknown='ignore',min_frequency=20))])
prep=ColumnTransformer([('num',num_pipe,num_cols),('cat',cat_pipe,cat_cols)],sparse_threshold=0.0)
X_train=prep.fit_transform(f.loc[train,X_cols])
X_val=prep.transform(f.loc[val,X_cols]);X_test=prep.transform(f.loc[test,X_cols])
assert np.isfinite(X_train).all() and np.isfinite(X_val).all() and np.isfinite(X_test).all()
TABULAR_IF_PARAMS=dict(n_estimators=300,max_samples=min(2048,len(X_train)),contamination='auto',n_jobs=-1)
small_model=IsolationForest(n_estimators=150,max_samples=TABULAR_IF_PARAMS['max_samples'],
                            contamination='auto',random_state=RANDOM_STATE,n_jobs=-1).fit(X_train)
tab_model=IsolationForest(random_state=RANDOM_STATE,**TABULAR_IF_PARAMS).fit(X_train)
val_corr=pd.Series(-small_model.score_samples(X_val)).corr(
    pd.Series(-tab_model.score_samples(X_val)),method='spearman')
print('150-vs-300-tree validation rank correlation (diagnostic, not accuracy):',round(val_corr,4))
f['tabular_if_score']=np.nan
f.loc[train,'tabular_if_score']=-tab_model.score_samples(X_train)
f.loc[val,'tabular_if_score']=-tab_model.score_samples(X_val)
f.loc[test,'tabular_if_score']=-tab_model.score_samples(X_test)
print('Tabular IF fitted on',len(X_train),'training lots,',X_train.shape[1],'encoded features.')
del small_model,X_train,X_val,X_test

# Part VII — Retrospective graph history (180 days, 30-day disclosure buffer)

**Source audit matters:** Prior lot deadlines are used as event times. `mw_offers.apply_date` validates whether an observed offer was submitted by that deadline, **not** whether the bidder identity was publicly available. Historical buyer–supplier features are therefore a **retrospective, not prospective**, extension.

A buyer–supplier relation is counted *once per distinct historical lot and supplier*, even when two lots have the same deadline. We exclude records with missing or post-deadline application timestamps and use only prior deadlines older than a 30-day buffer.

In [ ]:
all_hist=cohort[['id','buyer_id','cutoff_utc']].dropna(subset=['id','buyer_id','cutoff_utc']).drop_duplicates('id')
all_hist=all_hist.rename(columns={'cutoff_utc':'historical_deadline'})
o=offers[['trd_lots_id','sys_subjects_id','apply_date']].dropna(subset=['trd_lots_id','sys_subjects_id']).copy()
o['apply_utc']=pd.to_datetime(o['apply_date'],errors='coerce',utc=True)
o=o.rename(columns={'sys_subjects_id':'supplier_id'})
joined=o.merge(all_hist,left_on='trd_lots_id',right_on='id',how='inner',validate='many_to_one')
invalid_application=joined['apply_utc'].isna() | joined['apply_utc'].gt(joined['historical_deadline'])
source_audit=pd.DataFrame([{
    'joined_offer_rows':len(joined),
    'application_date_missing_or_after_deadline':int(invalid_application.sum()),
    'fraction_rejected':float(invalid_application.mean()) if len(joined) else np.nan,
    'offer_publication_time_verified':False
}])
display(source_audit)
source_audit.to_csv(MODEL_DIR/'07_offer_timestamp_audit.csv',index=False)
history=joined.loc[~invalid_application,['trd_lots_id','buyer_id','supplier_id','historical_deadline']].copy()
for col in ['buyer_id','supplier_id']:
    history[col]=history[col].astype('string')
history=history.drop_duplicates(['trd_lots_id','buyer_id','supplier_id'])
history=history.sort_values('historical_deadline').reset_index(drop=True)
print('Historical distinct lot-supplier participations:',len(history))
print('Not a proof that bidder identity was disclosed by the buffered date.')

In [ ]:
# Calendar-day rolling graph features, strictly before (prediction day - embargo).
from collections import Counter,defaultdict,deque
hist_dates=history['historical_deadline'].dt.floor('D').array
hbuyers=history['buyer_id'].to_numpy(dtype=str)
hsuppliers=history['supplier_id'].to_numpy(dtype=str)
lot_dates=f['cutoff_utc'].dt.floor('D')
query_dates=sorted(lot_dates.dropna().unique())
graph_cols=['g_buyer_offers_180d','g_distinct_suppliers_180d','g_repeat_partner_share_180d']
metrics={};edge_counts=defaultdict(Counter);window=deque();pos=0
for qd in query_dates:
    # A 180-day window ending 30 calendar days before the candidate deadline.
    upper=qd-pd.Timedelta(days=EMBARGO_DAYS)
    lower=qd-pd.Timedelta(days=LOOKBACK_DAYS)
    while pos<len(history) and hist_dates[pos]<upper:
        b,s=hbuyers[pos],hsuppliers[pos]
        edge_counts[b][s]+=1
        window.append((hist_dates[pos],b,s));pos+=1
    while window and window[0][0]<lower:
        _,b,s=window.popleft()
        edge_counts[b][s]-=1
        if edge_counts[b][s]<=0:del edge_counts[b][s]
        if not edge_counts[b]:del edge_counts[b]
    for buyer in f.loc[lot_dates.eq(qd),'buyer_id'].dropna().astype(str).unique():
        d=edge_counts.get(buyer,{})
        total=sum(d.values())
        metrics[(qd,buyer)]=(total,len(d),max(d.values())/total if total else 0.0)
metric_idx=pd.MultiIndex.from_arrays([lot_dates,f['buyer_id'].astype('string')])
metric_map=pd.DataFrame.from_dict(metrics,orient='index',columns=graph_cols)
metric_map.index=pd.MultiIndex.from_tuples(metric_map.index)
f[graph_cols]=metric_map.reindex(metric_idx).fillna(0).to_numpy(dtype='float32')
assert np.isfinite(f[graph_cols].to_numpy()).all()
assert (f['g_distinct_suppliers_180d']<=f['g_buyer_offers_180d']).all()
print('Buyer-supplier history coverage:',round(float((f['g_buyer_offers_180d']>0).mean()),3))
print('STRICT TEMPORAL BUFFER ≠ verified public bidder visibility. GRAPH MODEL IS RETROSPECTIVE.')

# Part VIII — Graph-enriched Isolation Forest

Train/test rows and model parameters are identical to the tabular model. Only the 180-day historical graph features are appended. This makes the comparison controlled but **conditional on the graph-availability assumption**.

In [ ]:
from sklearn.preprocessing import FunctionTransformer
all_num = num_cols + graph_cols
prep_g = ColumnTransformer([('num',clone(num_pipe),all_num),('cat',clone(cat_pipe),cat_cols)],sparse_threshold=.0)
Xg_train = prep_g.fit_transform(f.loc[train,all_num+cat_cols])
Xg_val = prep_g.transform(f.loc[val,all_num+cat_cols]);Xg_test = prep_g.transform(f.loc[test,all_num+cat_cols])
graph_model = IsolationForest(random_state=RANDOM_STATE,**TABULAR_IF_PARAMS).fit(Xg_train)
f['graph_if_score'] = np.nan
f.loc[train,'graph_if_score'] = -graph_model.score_samples(Xg_train)
f.loc[val,'graph_if_score'] = -graph_model.score_samples(Xg_val)
f.loc[test,'graph_if_score'] = -graph_model.score_samples(Xg_test)
print('Graph-enriched IF trained:',Xg_train.shape[1],'encoded features.')

# Part IX — Ranking comparison on the held-out test period

No ground-truth corruption labels are available. **Top-K agreement, score correlations and score stability are descriptive**, not Precision@K, sensitivity, specificity, or proof of detecting collusion. Comparisons are meaningful only within the same evaluated test lot population.

In [ ]:
score_cols=['red_flag_score','tabular_if_score','graph_if_score']
ft=f.loc[test,['lot_id','buy_id','buyer_id','cutoff_utc','amount_kzt','trade_method','category2']+
         flag_cols+score_cols+graph_cols].copy()
N=len(ft);assert N>0
rank_records=[];overlap=[]
for frac in [.01,.05,.10]:
    k=max(1,int(np.ceil(frac*N)))
    # Stable tie handling is especially important for the coarse integer red-flag ranking.
    tops={col:set(ft.sort_values([col,'lot_id'],ascending=[False,True],kind='stable')
                     .head(k)['lot_id']) for col in score_cols}
    for col in score_cols:
        # A coarse score may give many ties at the Top-K boundary; flag this explicitly.
        boundary=float(ft.sort_values([col,'lot_id'],ascending=[False,True],kind='stable').iloc[k-1][col])
        ties=int(ft[col].eq(boundary).sum())
        rank_records.append({'model':col,'top_fraction':frac,'lots_for_review':k,
                             'score_at_cutoff':boundary,'ties_at_cutoff_score':ties,
                             'ranking_tie_warning':ties>1})
    for i,a in enumerate(score_cols):
        for b in score_cols[i+1:]:
            intersection=len(tops[a]&tops[b])
            overlap.append({'top_fraction':frac,'model_a':a,'model_b':b,
                            'pair':a.replace('_score','')+' / '+b.replace('_score',''),
                            'intersection_count':intersection,'overlap_of_k':intersection/k,
                            'jaccard':intersection/(2*k-intersection),
                            'random_baseline_overlap':k/N})
rank_table=pd.DataFrame(rank_records);overlap_table=pd.DataFrame(overlap)
display(overlap_table.round(3));display(ft[score_cols].corr(method='spearman').round(3))
ft.sort_values('graph_if_score',ascending=False).to_csv(MODEL_DIR/'09_test_scores_RESTRICTED.csv',index=False)
rank_table.to_csv(MODEL_DIR/'09_review_budgets.csv',index=False)
overlap_table.to_csv(MODEL_DIR/'09_topk_overlap.csv',index=False)
fig,ax=plt.subplots(figsize=(10,4.5))
sns.barplot(data=overlap_table,x='top_fraction',y='overlap_of_k',hue='pair',ax=ax)
ax.set(title='Top-K ranking overlap (not fraud-detection precision)',
       xlabel='Share of test lots selected',ylabel='Shared fraction of the two top-K lists')
ax.legend(title='Compared approaches',bbox_to_anchor=(1.02,1),loc='upper left')
fig.tight_layout();fig.savefig(MODEL_DIR/'09_overlap.pdf',bbox_inches='tight');plt.show()

### 9.1 Sampling uncertainty in ranking agreement (not detection accuracy)
A test-only nonparametric bootstrap gives an interval for score correlation. This quantifies sampling variability of agreement, not model correctness. The random independent-ranking Top-K overlap baseline is approximately K/N.

In [ ]:
rng=np.random.default_rng(RANDOM_STATE)
a=ft['tabular_if_score'].to_numpy();b=ft['graph_if_score'].to_numpy()
rhos=[]
for _ in range(300):
    ii=rng.integers(0,len(ft),size=len(ft))
    rhos.append(spearmanr(a[ii],b[ii]).statistic)
observed=spearmanr(a,b).statistic
confidence=np.quantile(rhos,[.025,.975])
rank_ci=pd.DataFrame([{'pair':'Tabular IF vs Graph IF','spearman':observed,
                       'ci025':confidence[0],'ci975':confidence[1],
                       'meaning':'Ranking agreement only; no external validity labels'}])
display(rank_ci.round(4))
rank_ci.to_csv(MODEL_DIR/'09_spearman_bootstrap_ci.csv',index=False)

# Part X — Stability, graph-feature ablation, and controlled perturbations
We assess agreement across random seeds, sensitivity to removing one graph feature at a time (ablation *of the third approach*, not additional candidate models), and responses to synthetic **feature** changes. Synthetic changes do not simulate actual bid rigging or collusion.

In [ ]:
# Seed stability on test; no retraining of preprocessors on validation/test.
def seed_stability(Xtr,Xte,baseline_score,model_name):
    k=max(1,int(np.ceil(.05*len(Xte))))
    base_top=set(np.argpartition(-baseline_score,k-1)[:k])
    rows=[]
    for seed in SEEDS:
        m=IsolationForest(random_state=seed,**TABULAR_IF_PARAMS).fit(Xtr)
        s=-m.score_samples(Xte)
        top=set(np.argpartition(-s,k-1)[:k])
        rows.append({'model':model_name,'seed':seed,'top5_overlap':len(base_top&top)/k,
                     'spearman_to_baseline':pd.Series(s).corr(pd.Series(baseline_score),method='spearman')})
    return pd.DataFrame(rows)
Xt_train = prep.transform(f.loc[train,X_cols]);Xt_test=prep.transform(f.loc[test,X_cols])
stability=pd.concat([
    seed_stability(Xt_train,Xt_test,f.loc[test,'tabular_if_score'].to_numpy(),'tabular_if'),
    seed_stability(Xg_train,Xg_test,f.loc[test,'graph_if_score'].to_numpy(),'graph_if')
],ignore_index=True)
display(stability.groupby('model')[['top5_overlap','spearman_to_baseline']].agg(['mean','std']).round(3))
stability.to_csv(MODEL_DIR/'10_seed_stability.csv',index=False)

### 10.1 Graph-feature ablation (the same third method)
Remove each historical graph feature in turn, re-fit **the graph-enriched IF variant** on training observations, and measure changes in test rankings. We do not claim an improvement in fraud detection from unsupervised overlap alone.

In [ ]:
# Controlled ablation with unchanged train/test windows and model hyperparameters.
abl=[]
base_graph_test=f.loc[test,'graph_if_score'].to_numpy()
k=max(1,int(np.ceil(.05*test.sum())))
base_top=set(np.argpartition(-base_graph_test,k-1)[:k])
for excluded in graph_cols:
    remaining=[c for c in graph_cols if c!=excluded]
    ab_numeric=num_cols+remaining
    ab_prep=ColumnTransformer([('num',clone(num_pipe),ab_numeric),
                               ('cat',clone(cat_pipe),cat_cols)],sparse_threshold=0.0)
    ab_features=ab_numeric+cat_cols
    xa_train=ab_prep.fit_transform(f.loc[train,ab_features])
    xa_test=ab_prep.transform(f.loc[test,ab_features])
    ab_model=IsolationForest(random_state=RANDOM_STATE,**TABULAR_IF_PARAMS).fit(xa_train)
    ab_score=-ab_model.score_samples(xa_test)
    top=set(np.argpartition(-ab_score,k-1)[:k])
    abl.append({'removed_graph_feature':excluded,'rank_spearman_vs_full_graph':
                spearmanr(base_graph_test,ab_score).statistic,
                'top5_overlap_vs_full_graph':len(base_top & top)/k})
    del xa_train,xa_test,ab_model
ablation=pd.DataFrame(abl)
display(ablation.round(3))
ablation.to_csv(MODEL_DIR/'10_graph_feature_ablation.csv',index=False)

In [ ]:
# Paired synthetic perturbation on preselected test lots; data are NOT modified in place.
# This is a stress test for engineered features, NOT collusion recovery.
rng=np.random.default_rng(RANDOM_STATE)
Xt_test=prep.transform(f.loc[test,X_cols])
idx=rng.choice(len(Xt_test),size=min(1000,len(Xt_test)),replace=False)
original=f.loc[test,X_cols].iloc[idx].copy()
perturbed_raw=original.copy()
perturbed_raw['log_amount']=perturbed_raw['log_amount']+np.log(5)
perturbed_raw['duration_days']=perturbed_raw['duration_days'].clip(lower=0)/5
before=-tab_model.score_samples(prep.transform(original))
after=-tab_model.score_samples(prep.transform(perturbed_raw))
test_reference=f.loc[test,'tabular_if_score'].to_numpy()
threshold=float(np.quantile(test_reference,.95))
stress=pd.DataFrame({'before':before,'after':after,'difference':after-before,
                     'before_top5':before>=threshold,'after_top5':after>=threshold})
# Paired bootstrap over the chosen observations, not independent before/after groups.
boot=[];d=stress['difference'].to_numpy()
for _ in range(300):boot.append(np.mean(rng.choice(d,size=len(d),replace=True)))
ci=np.quantile(boot,[.025,.975])
stress_summary=pd.DataFrame([{'n':len(stress),'share_score_increased':(d>0).mean(),
    'mean_change':d.mean(),'mean_change_ci025':ci[0],'mean_change_ci975':ci[1],
    'top5_before':stress['before_top5'].mean(),'top5_after':stress['after_top5'].mean(),
    'top5_threshold_original_test':threshold}])
display(stress_summary.round(4))
stress_summary.to_csv(MODEL_DIR/'10_synthetic_stress_summary.csv',index=False)
fig,ax=plt.subplots(figsize=(8,4));ax.hist(stress['difference'].dropna().to_numpy(),bins=40,color='#4c78a8',edgecolor='white')
ax.axvline(0,color='black',linestyle='--')
ax.set(title='Controlled feature perturbation: change in anomaly score',xlabel='After − before')
fig.tight_layout();fig.savefig(MODEL_DIR/'10_synthetic_stress.pdf',bbox_inches='tight');plt.show()
del Xt_test

# Part XI — Blinded, independent expert review

The worksheet is a **sample for future review**, not validated labels. Two experts should separately examine the same cases using a written rubric (e.g., `0 = no issue identified`, `1 = investigate`, `NA = insufficient evidence`). Source documents should be reviewed, not merely anomaly scores. Reviewers must not receive model rankings. A separate restricted key preserves selection provenance. Keep the file private.

In [ ]:
review_ids=set();selection={}
for sc in score_cols:
    top=ft.sort_values([sc,'lot_id'],ascending=[False,True],kind='stable').head(max(1,int(np.ceil(.05*N))))['lot_id']
    chosen=top.sample(n=min(40,len(top)),random_state=RANDOM_STATE).tolist()
    for x in chosen:review_ids.add(x);selection.setdefault(x,[]).append(sc)
random_cases=ft['lot_id'].sample(n=min(100,N),random_state=RANDOM_STATE).tolist()
review_ids.update(random_cases)
review=ft.loc[ft['lot_id'].isin(review_ids),['lot_id','buy_id','cutoff_utc','amount_kzt','trade_method','category2']].copy()
review=review.sample(frac=1,random_state=RANDOM_STATE).reset_index(drop=True)
review.insert(0,'review_case_id',np.arange(1,len(review)+1))
review['reviewer_1_label']='';review['reviewer_2_label']='';review['adjudicated_label']=''
review['reviewer_notes']='';review['source_evidence']=''
review.to_csv(MODEL_DIR/'11_blinded_review_RESTRICTED.csv',index=False)
key=review[['review_case_id','lot_id']].copy()
key['selected_by']=key['lot_id'].map(lambda z:','.join(selection.get(z,['random'])))
key.to_csv(MODEL_DIR/'11_review_selection_key_RESTRICTED.csv',index=False)
print('Blind review cases generated:',len(review),'— labels are unfilled.')
display(review.head())

### 11.1 Label-processing gate (optional; zero invented labels)
After two independent reviewers fill in the restricted worksheet, copy it to `11_blinded_review_COMPLETED_RESTRICTED.csv`. This cell computes Cohen's κ **only when both reviewers have entered 0/1 labels**; otherwise it reports that review is pending. Adjudicated labels are required before any precision-style estimate, which is not automatically computed here because the case sampling design is not a uniform Top-K census.

In [ ]:
completed=MODEL_DIR/'11_blinded_review_COMPLETED_RESTRICTED.csv'
if not completed.exists():
    print('REVIEW STATUS: pending. No expert annotations, no Precision@K or accuracy claims.')
else:
    reviewed=pd.read_csv(completed)
    must=['review_case_id','reviewer_1_label','reviewer_2_label']
    assert set(must).issubset(reviewed.columns)
    from sklearn.metrics import cohen_kappa_score
    r1=pd.to_numeric(reviewed['reviewer_1_label'],errors='coerce')
    r2=pd.to_numeric(reviewed['reviewer_2_label'],errors='coerce')
    paired=r1.isin([0,1]) & r2.isin([0,1])
    if int(paired.sum())<2:
        print('REVIEW STATUS: fewer than two paired labels; no agreement estimate.')
    else:
        kappa=cohen_kappa_score(r1.loc[paired].astype(int),r2.loc[paired].astype(int))
        print('Paired annotated cases:',int(paired.sum()),'Cohen kappa:',round(float(kappa),3))
        print('Agreement describes reviewer concordance, not classifier precision.')

# Part XII — Reproducibility, version-disjoint sensitivity and research conclusions

**Do not report model accuracy without labels.** The following diagnostics quantify cross-split version-family overlap and run a second *component-disjoint sensitivity experiment*, using only families that occur in one split. The primary full-period analysis retains the original 115,177 lots, while the sensitivity experiment has fewer lots by design.

In [ ]:
# Full 12-month results remain the PRIMARY retrospective experiment.
summary_results=pd.DataFrame([
 {'measure':'total_pilot_lots','value':len(f)},
 {'measure':'train_lots','value':int(train.sum())},
 {'measure':'validation_lots','value':int(val.sum())},
 {'measure':'test_lots','value':int(test.sum())},
 {'measure':'historical_graph_coverage_fraction','value':float((f['g_buyer_offers_180d']>0).mean())},
 {'measure':'direct_cross_split_parent_links','value':len(cross_split_links)},
 {'measure':'blind_review_cases_pending_labels','value':len(review)}
])
display(summary_results)
summary_results.to_csv(MODEL_DIR/'12_research_summary.csv',index=False)
from importlib.metadata import version,PackageNotFoundError
versions={'python':sys.version.split()[0]}
for pkg in ['pandas','numpy','pyarrow','scikit-learn','matplotlib','seaborn','scipy']:
    try:versions[pkg]=version(pkg)
    except PackageNotFoundError:versions[pkg]='unavailable'
metadata={'input_manifest':manifest,'packages':versions,
 'parameters':{'seed':RANDOM_STATE,'seeds':SEEDS,'lookback_days':LOOKBACK_DAYS,
               'embargo_days':EMBARGO_DAYS,'lot_limit_waived':True,
               'model_methods':['five_heuristic_flags','tabular_isolation_forest','graph_enriched_isolation_forest']},
 'limitations':['Retrospective export — publication/amendment history unavailable',
                'No confirmed bid-opening/public bidder-disclosure timestamp',
                'No independently verified corruption labels',
                'Version-family disjoint sensitivity performed separately',
                'All rankings are candidate screens, not accusations']}
(MODEL_DIR/'12_run_metadata.json').write_text(json.dumps(metadata,ensure_ascii=False,indent=2,default=str),encoding='utf-8')
print('Research artifacts:',MODEL_DIR)

### 12.1 Strict **version-component-disjoint** sensitivity analysis
Using connected components from the entire reissue graph, **exclude all components represented in more than one split**, then re-fit tabular and graph-enriched IF on the remaining training lots. The 5-flag baseline uses the training-derived cutoffs already defined. The original 115,177-lot full sample is still the primary descriptive experiment.

The sensitivity analysis tests how much rankings change when version families are segregated, but does **not** resolve unavailable point-in-time source history.

In [ ]:
# Refresh the version-component marker after feature-frame sorting/merges.
version_audit=pilot[['lot_id','version_component_id','cross_split_component']].copy()
fs=f.merge(version_audit,on='lot_id',how='left',validate='one_to_one')
assert len(fs)==len(f)
strict=~fs['cross_split_component'].fillna(False).to_numpy()
strict_train=strict & fs['split'].eq('train').to_numpy()
strict_test=strict & fs['split'].eq('test').to_numpy()
strict_val=strict & fs['split'].eq('validation').to_numpy()
assert strict_train.sum()>0 and strict_test.sum()>0
assert fs.loc[strict,'version_component_id'].groupby(fs.loc[strict,'version_component_id']).size().sum()==strict.sum()
assert fs.loc[strict].groupby('version_component_id')['split'].nunique().le(1).all()
print('Strict counts:',{'train':int(strict_train.sum()),'validation':int(strict_val.sum()),
                        'test':int(strict_test.sum()),'dropped_total':int((~strict).sum())})
strict_results=[]
for model_name,continuous in [('tabular_if',num_cols),('graph_if',num_cols+graph_cols)]:
    pp=ColumnTransformer([('num',clone(num_pipe),continuous),
                          ('cat',clone(cat_pipe),cat_cols)],sparse_threshold=0.0)
    columns=continuous+cat_cols
    xtr=pp.fit_transform(fs.loc[strict_train,columns])
    xte=pp.transform(fs.loc[strict_test,columns])
    fresh=IsolationForest(random_state=RANDOM_STATE,**TABULAR_IF_PARAMS).fit(xtr)
    new_score=-fresh.score_samples(xte)
    orig_score=fs.loc[strict_test,'tabular_if_score' if model_name=='tabular_if' else 'graph_if_score'].to_numpy()
    k_strict=max(1,int(np.ceil(.05*strict_test.sum())))
    top_orig=set(np.argpartition(-orig_score,k_strict-1)[:k_strict])
    top_new=set(np.argpartition(-new_score,k_strict-1)[:k_strict])
    strict_results.append({'model':model_name,'strict_train_lots':int(strict_train.sum()),
       'strict_test_lots':int(strict_test.sum()),'spearman_vs_original_restricted':spearmanr(orig_score,new_score).statistic,
       'top5_overlap_vs_original_restricted':len(top_orig&top_new)/k_strict})
    del xtr,xte,fresh
strict_report=pd.DataFrame(strict_results)
display(strict_report.round(3))
strict_report.to_csv(MODEL_DIR/'12_component_disjoint_sensitivity.csv',index=False)

### 12.2 Final interpretation checklist

* **Supported:** auditable data counts, full 12-month chronology, three implemented scoring methods, descriptive agreement, uncertainty, seed stability, graph-feature ablation, synthetic-feature stress, and connected-component sensitivity.
* **Not supported:** claims that flagged lots are fraudulent; verified prospective feature availability; recall, precision, true-positive rate, or a causal advantage from graph structure without external expert/legal labels.
* **Next external dependency:** obtain dated public release/audit records for the portal and independent, double-reviewed procurement case labels. Until then, all conclusions should be phrased as *retrospective anomaly screening*.

Figures and tables can be included in a scientific paper with these qualifications. Publish code and aggregate counts only; keep raw and row-level review data restricted.

---
# Part XIII — Specification-aligned extension (v3)

The original 12-part pipeline above is retained for comparison, **but its published scores are exploratory and should not be reused as v3 model scores**. The cells below implement the predeclared six-point hyperparameter grid, 10 seeds, supplier co-bidding graph with weekly snapshots, two planted-pattern experiments, paired bootstrap, sensitivity tests, blinded reviewer workbook, machine-readable output tables and leakage assertions. Run **all** cells in order.

**Availability gate:** the export has bid-submission timestamps but no independently established public bid-opening time. Supplier identities and winners of the *current* lot are consequently unavailable at the assumed deadline. The extended graph and planted scenarios explicitly require these identities and are **conditional retrospective diagnostics only**; they cannot be characterized as leak-free prospective detection. Historical winner publication times are also unverified. Rotation is therefore a **simulated historical winner assignment**, not verified history. Report this deviation to the article authors.

**Performance:** weekly graph snapshots across 365 days can take substantial RAM and computation. Results are exported only if cells finish. Results from v2 are not silently relabeled as v3.

In [ ]:
import networkx as nx
from itertools import combinations
from collections import defaultdict
from scipy.stats import rankdata
from sklearn.metrics import cohen_kappa_score
import hashlib, platform, time
from datetime import datetime, timezone

V3_DIR=ROOT/'outputs'/'spec_aligned_v3'
V3_DIR.mkdir(parents=True,exist_ok=True)
V3_SEEDS=list(range(42,52))
V3_KS=(.01,.05,.10)
V3_GRID=[(trees,draw) for trees in (100,200) for draw in (128,256,512)]
V3_BOOT=2000
V3_SYNTH_SEED=20261013
V3_EMBARGO=30  # Explicit conservative proxy, *not* proof of disclosure.
V3_DEPENDENCY='conditional_retrospective_bidder_visibility'

# Strict strings: do not let a 1.0 float ID become a different supplier than '1'.
def canonical_id(series):
    ss=series.astype('string').str.strip()
    ss=ss.str.replace(r'\.0$','',regex=True)
    return ss.mask(ss.isin(['','nan','None','<NA>']))

def topids(ids,scores,share=.05):
    n=max(1,int(np.ceil(len(ids)*share)))
    order=np.lexsort((np.asarray(ids,dtype=str),-np.asarray(scores,dtype=float)))
    return set(np.asarray(ids,dtype=str)[order[:n]])

def jac(a,b):
    return len(a&b)/len(a|b) if (a or b) else np.nan

def stable_rank(ids,score):
    ix=np.lexsort((np.asarray(ids,dtype=str),-np.asarray(score,dtype=float)))
    ranks=np.empty(len(ix),dtype=np.int32);ranks[ix]=np.arange(1,len(ix)+1)
    return ranks

# Source timeline: historical lot bidding deadlines are *not* disclosure timestamps.
hv=history[['trd_lots_id','historical_deadline','supplier_id','buyer_id']].copy()
hv['lot']=canonical_id(hv.trd_lots_id);hv['supplier']=canonical_id(hv.supplier_id)
hv['buyer']=canonical_id(hv.buyer_id)
hv=hv.dropna(subset=['lot','supplier','historical_deadline']).drop_duplicates(['lot','supplier'])
hv['day']=hv.historical_deadline.dt.floor('D')
hv=hv.sort_values('historical_deadline')
# Current-lot participants below are ONLY used in a retrospective experiment.
participant_map=hv.groupby('lot',observed=True)['supplier'].agg(lambda z:tuple(sorted(set(z)))).to_dict()
# Since source rows may be rejected due to bad timing, do not invent any absent bidders.
f['v3_lot_key']=canonical_id(f.lot_id)
f['v3_participants']=f['v3_lot_key'].map(participant_map).map(lambda z:z if isinstance(z,tuple) else ())
f['v3_n_observed_bidders']=f.v3_participants.map(len)
print('NetworkX',nx.__version__,'observed 2+ bidder test lots:',int((f.loc[test,'v3_n_observed_bidders']>=2).sum()))


## 13.1 Weekly historical supplier network

Build supplier–lot incidence using earlier bidding deadlines; project co-participation to supplier–supplier edges. The weekly snapshot is frozen at the beginning of the week with a 30-day safety buffer, thus excluding same-week and current-lot history. Community IDs are never model inputs. Weekly approximation and the 30-day embargo are documented changes from a per-event 180-day exact graph. Missing current participants produce missing supplier attributes and explicit flags, *not zeros interpreted as true no-risk*.

In [ ]:
from functools import lru_cache

# Only the empirical observed-bidder coverage subset has retrospective supplier aggregates.
# Snapshot computation is cached. Bidders not observed at prediction time: conditional analysis.
@lru_cache(maxsize=100)
def weekly_snapshot(week_string,lookback=180):
    week=pd.Timestamp(week_string)
    upper=week-pd.Timedelta(days=V3_EMBARGO)
    lower=week-pd.Timedelta(days=lookback)
    h=hv.loc[(hv.historical_deadline>=lower)&(hv.historical_deadline<upper)]
    groups=h.groupby('lot',sort=False,observed=True)
    G=nx.Graph();supply_lots=defaultdict(set);win_by_buyer=defaultdict(list)
    # Historical winner NOT inferred: rotation_score is unavailable on unmodified records.
    for lot,g in groups:
        ppl=sorted(set(g.supplier))
        for s in ppl:
            G.add_node(s);supply_lots[s].add(lot)
        for a,b in combinations(ppl,2):
            if G.has_edge(a,b):G[a][b]['weight']+=1
            else:G.add_edge(a,b,weight=1)
    # Deterministic Louvain as specified (NetworkX >=3.3).
    communities=nx.community.louvain_communities(G,weight='weight',seed=42) if G.number_of_nodes() else []
    lookup={};sizes={};densities={}
    for idx,community in enumerate(communities):
        sizes[idx]=len(community)
        densities[idx]=nx.density(G.subgraph(community)) if len(community)>1 else 0.0
        for s in community:lookup[s]=idx
    return G,supply_lots,lookup,sizes,densities

def network_features(participants,week,window=180, injected=None):
    cols=['deg_mean','deg_max','w_deg_mean','w_deg_max','pair_cobid_max','pair_jaccard_max',
          'rotation_score','same_comm_share','comm_size_mean','comm_size_max',
          'comm_density_mean','comm_density_max','buyer_top1_share','graph_missing']
    if not participants:
        return dict.fromkeys(cols,np.nan)|{'graph_missing':1.0}
    G,lots_by_s,comm,sizes,densities=weekly_snapshot(str(week),window)
    # Injection for tests: isolated copy, never mutate cached historical snapshot.
    if injected:
        G=G.copy();lots_by_s={k:set(v) for k,v in lots_by_s.items()}
        for lot_id,ppl in injected['historical_lots']:
            for s in ppl:G.add_node(s);lots_by_s.setdefault(s,set()).add(lot_id)
            for a,b in combinations(ppl,2):
                if G.has_edge(a,b):G[a][b]['weight']+=1
                else:G.add_edge(a,b,weight=1)
        # Recompute communities only for injected subset's connected graph.
        cm=nx.community.louvain_communities(G,weight='weight',seed=42)
        comm={s:i for i,c in enumerate(cm) for s in c}
        sizes={i:len(c) for i,c in enumerate(cm)}
        densities={i:nx.density(G.subgraph(c)) for i,c in enumerate(cm)}
    deg=[float(G.degree(s)) if s in G else 0. for s in participants]
    wdeg=[float(G.degree(s,weight='weight')) if s in G else 0. for s in participants]
    pvals=[];jvals=[]
    for a,b in combinations(participants,2):
        pvals.append(float(G[a][b]['weight']) if G.has_edge(a,b) else 0.0)
        x=lots_by_s.get(a,set());y=lots_by_s.get(b,set())
        jvals.append(len(x&y)/len(x|y) if x or y else 0.)
    cs=[comm[s] for s in participants if s in comm]
    same=max((cs.count(c) for c in set(cs)),default=0)/len(participants)
    return {
       'deg_mean':float(np.mean(deg)),'deg_max':max(deg),
       'w_deg_mean':float(np.mean(wdeg)),'w_deg_max':max(wdeg),
       'pair_cobid_max':max(pvals,default=np.nan),
       'pair_jaccard_max':max(jvals,default=np.nan),
       'rotation_score':float(injected.get('rotation',np.nan)) if injected else np.nan,
       'same_comm_share':same,
       'comm_size_mean':float(np.mean([sizes.get(comm.get(s),0) for s in participants])),
       'comm_size_max':max([sizes.get(comm.get(s),0) for s in participants]),
       'comm_density_mean':float(np.mean([densities.get(comm.get(s),0.) for s in participants])),
       'comm_density_max':max([densities.get(comm.get(s),0.) for s in participants]),
       'buyer_top1_share':np.nan, # historical winner disclosure unverified — NOT substituted
       'graph_missing':0.0
    }

f['v3_week']=f.cutoff_utc.dt.floor('D')-pd.to_timedelta(f.cutoff_utc.dt.dayofweek,unit='D')
V3_GRAPH_COLS=list(network_features((),str(pd.Timestamp('2026-01-05',tz='UTC'))).keys())
V3_VALID_GRAPH_COLS=[c for c in V3_GRAPH_COLS if c not in ('rotation_score','buyer_top1_share')]
# Compute once per (week, participant set), with candidate-specific strict exclusion by earlier week.
feature_cache={}
new_graph=[]
for i,(participants,week) in enumerate(zip(f.v3_participants,f.v3_week)):
    key=(tuple(participants),str(week))
    if key not in feature_cache:
        feature_cache[key]=network_features(participants,week)
    new_graph.append(feature_cache[key])
    if i and i%25000==0:print('Graph lots:',i,'/',len(f),'unique queries:',len(feature_cache))
fg=pd.DataFrame(new_graph,index=f.index)
for c in V3_GRAPH_COLS:f['v3_'+c]=pd.to_numeric(fg[c],errors='coerce')
print('Calculated retrospective graph features:',V3_GRAPH_COLS)
print('Not available / not evaluated:', ['rotation_score','buyer_top1_share'])


## 13.2 Predeclared validation selection, 10 seeds and common IF configuration

Validation chooses the shared `(n_estimators, max_samples)` maximizing mean pairwise Jaccard@5% across **both** variants. Evaluations on test use the same 10 seeds, 42–51. No labels are used for model selection. The two winner-dependent graph fields remain missing; only the computable subset is used and this deviation is logged.

In [ ]:
from sklearn.base import clone
V3_GRAPH_NUM=['v3_'+c for c in V3_VALID_GRAPH_COLS]
V3_TAB_NUM=num_cols.copy();V3_CATS=cat_cols.copy()

def make_matrix(columns):
    transformer=ColumnTransformer([
        ('numeric',clone(num_pipe),columns),
        ('categorical',clone(cat_pipe),V3_CATS)
    ],sparse_threshold=0.0)
    fields=columns+V3_CATS
    A=transformer.fit_transform(f.loc[train,fields]);B=transformer.transform(f.loc[val,fields]);C=transformer.transform(f.loc[test,fields])
    return transformer,A,B,C

v3_prep_t,v3_train_t,v3_val_t,v3_test_t=make_matrix(V3_TAB_NUM)
v3_prep_g,v3_train_g,v3_val_g,v3_test_g=make_matrix(V3_TAB_NUM+V3_GRAPH_NUM)

# Stable case sorting on lot ID breaks ties in exactly the same way for every seed.
val_ids=f.loc[val,'lot_id'].to_numpy(str)
test_ids=f.loc[test,'lot_id'].to_numpy(str)

def all_seed_scores(Xtrain,Xhold,trees,samples):
    return {seed:-IsolationForest(n_estimators=trees,max_samples=samples,
               contamination='auto',random_state=seed,n_jobs=-1).fit(Xtrain).score_samples(Xhold)
            for seed in V3_SEEDS}

def mean_pairwise_jaccard(ids,scored,share=.05):
    sets={s:topids(ids,sc,share) for s,sc in scored.items()}
    return float(np.mean([jac(sets[a],sets[b]) for a,b in combinations(sorted(sets),2)]))

tuning=[]
for trees,samples in V3_GRID:
    scores_t=all_seed_scores(v3_train_t,v3_val_t,trees,samples)
    scores_g=all_seed_scores(v3_train_g,v3_val_g,trees,samples)
    jt=mean_pairwise_jaccard(val_ids,scores_t)
    jg=mean_pairwise_jaccard(val_ids,scores_g)
    tuning.append({'n_estimators':trees,'max_samples':samples,'jaccard_tabular':jt,
                   'jaccard_graph':jg,'mean_jaccard':(jt+jg)/2})
v3_tuning=pd.DataFrame(tuning).sort_values(['mean_jaccard','n_estimators','max_samples'],ascending=[False,True,True])
v3_tuning.to_csv(V3_DIR/'tuning_validation.csv',index=False)
display(v3_tuning.round(4))
chosen=v3_tuning.iloc[0]
V3_TREES=int(chosen.n_estimators);V3_SAMPLES=int(chosen.max_samples)
print('Chosen SHARED parameters:',V3_TREES,V3_SAMPLES)

v3_scores_t=all_seed_scores(v3_train_t,v3_test_t,V3_TREES,V3_SAMPLES)
v3_scores_g=all_seed_scores(v3_train_g,v3_test_g,V3_TREES,V3_SAMPLES)
v3_rankings=pd.DataFrame({'lot_id':test_ids,'red_flag_score':f.loc[test,'red_flag_score'].to_numpy()})
v3_rankings['red_flag_rank']=stable_rank(test_ids,v3_rankings.red_flag_score)
for model,collection in [('tabular',v3_scores_t),('graph',v3_scores_g)]:
    for seed,sc in collection.items():
        v3_rankings[f'{model}_score_seed_{seed}']=sc
        v3_rankings[f'{model}_rank_seed_{seed}']=stable_rank(test_ids,sc)
v3_rankings.to_csv(V3_DIR/'rankings_test_RESTRICTED.csv',index=False)
print('All 10-seed test rankings saved:',len(v3_rankings),'lots')


## 13.3 Exact 45-pair seed stability; K = 1%, 5%, 10%; graph window diagnostics

Measure overlap using true **Jaccard** `|A∩B| / |A∪B|` rather than overlap divided by K. The 90/180/365-day comparison is computed on the restricted coverage-eligible test set; because the 365-day source coverage is not necessarily complete, the eligible population and any shortage are explicitly reported. Graph windows are recomputed; fitted preprocessors/models remain fixed for a *feature-perturbation* comparison, and this is not a new training run.

In [ ]:
seed_rows=[];matrices=[]
for model,sc in [('tabular',v3_scores_t),('graph',v3_scores_g)]:
    for frac in V3_KS:
        sets={s:topids(test_ids,arr,frac) for s,arr in sc.items()}
        for a,b in combinations(V3_SEEDS,2):
            seed_rows.append({'model':model,'K':frac,'seed_a':a,'seed_b':b,'jaccard':jac(sets[a],sets[b])})
    matrix=np.array([[jac(topids(test_ids,sc[a],.05),topids(test_ids,sc[b],.05)) for b in V3_SEEDS] for a in V3_SEEDS])
    matrices.append((model,matrix))
v3_seed_pairs=pd.DataFrame(seed_rows)
assert len(v3_seed_pairs)==2*3*45
v3_seed_pairs.to_csv(V3_DIR/'seed_jaccard_45pairs.csv',index=False)

fig,axes=plt.subplots(1,2,figsize=(12,5),constrained_layout=True)
for ax,(model,matrix) in zip(axes,matrices):
    im=ax.imshow(matrix,vmin=0,vmax=1,cmap='viridis')
    ax.set_xticks(range(10),V3_SEEDS,rotation=45);ax.set_yticks(range(10),V3_SEEDS)
    ax.set_xlabel(f'{model}: seed');ax.set_ylabel('Seed')
fig.colorbar(im,ax=axes,label='Top-5% Jaccard')
fig.savefig(V3_DIR/'fig_seed_jaccard.pdf',bbox_inches='tight');plt.show()

# Window check, only on test cohort with 365 complete observed days before snapshot.
coverage_begin=hv.historical_deadline.min()
full365=(f.loc[test,'v3_week']-pd.Timedelta(days=365)>=coverage_begin)
# Offer visibility is unverified even for this coverage-filtered sample.
window_idx=np.flatnonzero(full365.to_numpy())
window_report=[]
base=v3_scores_g[42]
for days in (90,180,365):
    if not len(window_idx):
        window_report.append({'window_days':days,'eligible_lots':0,'jaccard_vs_180':np.nan})
        continue
    data=f.loc[test].iloc[window_idx]
    modified=pd.DataFrame([network_features(p,w,days) for p,w in zip(data.v3_participants,data.v3_week)])
    inputs=data[V3_TAB_NUM+V3_GRAPH_NUM+V3_CATS].copy().reset_index(drop=True)
    for c in V3_VALID_GRAPH_COLS:inputs['v3_'+c]=modified[c].to_numpy()
    X=v3_prep_g.transform(inputs)
    # Refit ONLY on the pre-selected historical training matrix, not on modified test data.
    fixed=IsolationForest(n_estimators=V3_TREES,max_samples=V3_SAMPLES,
             random_state=42,contamination='auto',n_jobs=-1).fit(v3_train_g)
    ss=-fixed.score_samples(X)
    subset_ids=test_ids[window_idx]
    window_report.append({'window_days':days,'eligible_lots':len(window_idx),
       'jaccard_vs_180':jac(topids(subset_ids,base[window_idx],.05),topids(subset_ids,ss,.05)),
       'note':'fixed-model test feature sensitivity, not retrained IF'})
window_table=pd.DataFrame(window_report)
window_table.to_csv(V3_DIR/'window_sensitivity.csv',index=False)
display(window_table)
fig,ax=plt.subplots(figsize=(6,4))
ax.plot(window_table.window_days,window_table.jaccard_vs_180,'o-')
ax.set(xlabel='Graph lookback window (days)',ylabel='Top-5% Jaccard to original Graph IF',ylim=(0,1))
fig.tight_layout();fig.savefig(V3_DIR/'fig_window_sensitivity.pdf');plt.show()


## 13.4 Planted co-bidding / winner rotation: conditional synthetic diagnostic

For each pattern, choose 50 test lots with ≥2 observed bids (seed 20261013). Inject three **synthetic** suppliers into copies of six preceding historical lots (within the allowed window, not the target lot itself), plus into the corresponding target-lot bidder set. Co-bidding uses no winner labels; rotation assigns cyclic synthetic historical winners and sets a synthetic rotation score. The scenario does not modify the original Parquet files or retrain models. Since historical winner publication timestamps are missing, rotation is **not an evaluation of observed winner history**. All three methods are rescored for every test lot, leaving unchanged lots' features the same; the tabular model and baseline necessarily have unchanged features under a graph-only intervention, making the comparison deliberately favorable to Graph IF. The fraction of planted lots in each full-test Top-K is recovery@K, not precision against true violations.

In [ ]:
eligible=np.flatnonzero((f.loc[test,'v3_n_observed_bidders']>=2).to_numpy())
if len(eligible)<50:
    raise ValueError(f'Synthetic study requires >=50 test lots with 2+ observed offers; found {len(eligible)}')
rng=np.random.default_rng(V3_SYNTH_SEED)
target_pos=np.sort(rng.choice(eligible,size=50,replace=False))
V3_SYNTH_IDS=test_ids[target_pos]
# Unique IDs so synthetic edges cannot accidentally collide with real suppliers.
synthetic_suppliers=('SIMULATED_SUP_001','SIMULATED_SUP_002','SIMULATED_SUP_003')

# Freeze seed-42 fitted models once and make transformations on copies only.
fixed_graph=IsolationForest(n_estimators=V3_TREES,max_samples=V3_SAMPLES,
    contamination='auto',random_state=42,n_jobs=-1).fit(v3_train_g)
fixed_tab=IsolationForest(n_estimators=V3_TREES,max_samples=V3_SAMPLES,
    contamination='auto',random_state=42,n_jobs=-1).fit(v3_train_t)
original_g=v3_scores_g[42].copy();original_t=v3_scores_t[42].copy()

def run_planted(pattern):
    hypothetical=f.loc[test].copy().reset_index(drop=True)
    for pos in target_pos:
        row=hypothetical.iloc[pos]
        # Six synthetic historical events end >30d before beginning of candidate week.
        cutoff_week=row.v3_week
        event_days=[cutoff_week-pd.Timedelta(days=d) for d in (40,60,80,100,120,140)]
        synthetic_hist=[('SIM_HISTORY_'+str(pos)+'_'+str(k),synthetic_suppliers)
                        for k in range(6)]
        synthetic_observed=tuple(sorted(set(row.v3_participants)|set(synthetic_suppliers)))
        injection={'historical_lots':synthetic_hist,
                   'rotation':1.0 if pattern=='rotation' else np.nan,
                   'event_days':event_days}
        feats=network_features(synthetic_observed,cutoff_week,180,injected=injection)
        for feature in V3_VALID_GRAPH_COLS:
            hypothetical.at[pos,'v3_'+feature]=feats[feature]
        # A synthetic rotation indicator is not a fitted model input in the v3 graph,
        # because true historical winners cannot be certified. Do NOT claim model
        # sensitivity to rotation merely by setting this absent field.
    Xg=v3_prep_g.transform(hypothetical[V3_TAB_NUM+V3_GRAPH_NUM+V3_CATS])
    updated_g=-fixed_graph.score_samples(Xg)
    # Tabular and baseline re-scored on unchanged tabular fields for fair placebo.
    Xt=v3_prep_t.transform(hypothetical[V3_TAB_NUM+V3_CATS])
    updated_t=-fixed_tab.score_samples(Xt)
    outputs={'red_flag':hypothetical.red_flag_score.to_numpy(dtype=float),
             'tabular':updated_t,'graph':updated_g}
    return outputs

synthetic_outputs={}
synthetic_rows=[]
for pattern in ('co_bidding','rotation'):
    outputs=run_planted(pattern)
    synthetic_outputs[pattern]=outputs
    for method,scores in outputs.items():
        for K in V3_KS:
            recovered=len(set(V3_SYNTH_IDS)&topids(test_ids,scores,K))
            synthetic_rows.append({'pattern':pattern,'model':method,'K':K,
                                   'recovered':recovered,'n_planted':50,'recovery_at_k':recovered/50})
v3_recovery=pd.DataFrame(synthetic_rows)
v3_recovery.to_csv(V3_DIR/'synthetic_recovery.csv',index=False)
display(v3_recovery)
# Explicit identifiability warning: rotation is not statistically identifiable in
# this model, because verified historical winner labels are unavailable.
print('IMPORTANT: rotation winner order is NOT a model input; rotation recovery is a co-bidding-only diagnostic.')
fig,axes=plt.subplots(1,2,figsize=(11,4),sharey=True)
for ax,(pattern,grp) in zip(axes,v3_recovery.groupby('pattern',sort=False)):
    for method,r in grp.groupby('model'):
        ax.plot(100*r.K,r.recovery_at_k,'o-',label=method)
    ax.set(xlabel='K (% of test lots)',ylabel='Planted-lot recovery@K',ylim=(0,1))
    ax.text(.03,.95,pattern,transform=ax.transAxes,va='top')
axes[0].legend(frameon=False)
fig.tight_layout();fig.savefig(V3_DIR/'fig_recovery_k.pdf');plt.show()


## 13.5 Paired bootstrap, ablation and reproducibility outputs

Bootstrap resamples lots **with replacement**, re-ranks within each bootstrap sample, and computes paired model contrasts on the *same draws* (2000 iterations, fixed seed). Repeated case IDs are treated as independently resampled observations; this estimates lot-level uncertainty and does not resolve within-buyer or linked-version dependence. 97.5% two-sided intervals correspond to the `1.25/98.75` percentiles, while 95% intervals use `2.5/97.5` percentiles. Both contrasts are exploratory given missing verified winner-publication times and synthetic injection favoring the graph method.

In [ ]:
# Always include the unmodified original test scores in the machine-readable output.
assert np.allclose(original_g,v3_scores_g[42]) and np.allclose(original_t,v3_scores_t[42])
B=V3_BOOT;boot_rng=np.random.default_rng(V3_SYNTH_SEED)
contrasts=[]
for pattern,outs in synthetic_outputs.items():
    for contrast,a,b in [('graph_minus_tabular','graph','tabular'),('graph_minus_redflag','graph','red_flag')]:
        vals=[]
        planted=np.isin(test_ids,V3_SYNTH_IDS)
        for iteration in range(B):
            take=boot_rng.integers(0,len(test_ids),len(test_ids))
            # Multiplicity must be preserved; use integer positions, not a set of identifiers.
            ka=max(1,int(np.ceil(.05*len(take))))
            sa=outs[a][take];sb=outs[b][take];yt=planted[take]
            ids_boot=np.arange(len(take)).astype(str)
            oa=np.lexsort((ids_boot,-sa))[:ka]
            ob=np.lexsort((ids_boot,-sb))[:ka]
            denom=max(1,int(yt.sum()))
            vals.append((yt[oa].sum()-yt[ob].sum())/denom)
        vals=np.array(vals)
        observed=(len(topids(test_ids,outs[a],.05)&set(V3_SYNTH_IDS))-
                  len(topids(test_ids,outs[b],.05)&set(V3_SYNTH_IDS)))/50
        contrasts.append({'pattern':pattern,'contrast':contrast,'K':.05,
            'estimate':observed,'ci95_low':np.quantile(vals,.025),'ci95_high':np.quantile(vals,.975),
            'ci97_5_low':np.quantile(vals,.0125),'ci97_5_high':np.quantile(vals,.9875),'n_boot':B})
v3_contrasts=pd.DataFrame(contrasts)
v3_contrasts.to_csv(V3_DIR/'tab_contrasts.csv',index=False)

main=[]
for model,coll in [('tabular',v3_scores_t),('graph',v3_scores_g)]:
    for K in V3_KS:
        jj=v3_seed_pairs.loc[(v3_seed_pairs.model==model)&(v3_seed_pairs.K==K),'jaccard']
        main.append({'model':model,'K':K,'metric':'seed_pair_jaccard',
                     'mean':jj.mean(),'sd':jj.std(),'n_seed_pairs':len(jj)})
for _,r in v3_recovery.iterrows():
    main.append({'model':r.model,'K':r.K,'metric':'synthetic_'+r.pattern+'_recovery',
                 'mean':r.recovery_at_k,'sd':np.nan,'n_seed_pairs':np.nan})
v3_main=pd.DataFrame(main);v3_main.to_csv(V3_DIR/'tab_main.csv',index=False)

# Required coverage/missingness limitation table.
coverage=f.groupby('split').agg(lots=('lot_id','size'),observed_bids_ge2=('v3_n_observed_bidders',lambda x:int((x>=2).sum())),
                                 no_observed_bidder=('v3_n_observed_bidders',lambda x:int((x==0).sum()))).reset_index()
coverage['supplier_visibility_verified']=False
coverage.to_csv(V3_DIR/'tab_data_account.csv',index=False)

def sha256(path):
    h=hashlib.sha256()
    with open(path,'rb') as fd:
        for piece in iter(lambda:fd.read(2**20),b''):h.update(piece)
    return h.hexdigest()

runlog={'timestamp_utc':datetime.now(timezone.utc).isoformat(),'mode':'conditional_retrospective',
        'data_manifest_sha256':sha256(manifest_path) if manifest_path.exists() else None,
        'seeds':V3_SEEDS,'synthetic_seed':V3_SYNTH_SEED,'bootstrap_draws':B,
        'model_selection_grid':V3_GRID,'selected_parameters':[V3_TREES,V3_SAMPLES],
        'cutoff':'bid_deadline_proxy','offer_visibility_proven':False,
        'winner_publication_verified':False,
        'winner_rotation_score_used_in_model':False,
        'window_lookback_days':[90,180,365], 'history_embargo_days':V3_EMBARGO,
        'source_data_in_public_repo':False,'host':platform.platform(),
        'processor':platform.processor(),
        'python':platform.python_version(),
        'limitations':['Data-snapshot provenance unavailable','Current participants observed retrospectively only',
            'No full verified historical winner feature block','Synthetic rotation score excluded from model',
            'Bidder graph requires portal disclosure-timestamp verification','100k lot cap waived']}
(V3_DIR/'run_log.json').write_text(json.dumps(runlog,ensure_ascii=False,indent=2,default=str),encoding='utf-8')
print('Completed computable v3 extension outputs in',V3_DIR)
print('Not claimed complete: full winning-rotation signal, expert review ground truth, point-in-time public visibility.')


## Part XIV — Submission gate / deviations from the specification

**Completed only after executing the extension:** validation grid, 10-seed comparison, 45 pairwise seed Jaccards, full-test rankings, conditional weekly co-bidding features, partial graph ablations, 90/180/365-day window diagnostics, co-bidding planted perturbations, lot-level paired bootstrap, machine-readable CSV/PDF outputs and run metadata.

**Not fully compliant:** (i) no confirmed public bid-opening time; (ii) historical winner decisions lack certified publication times, so rotation and buyer concentration cannot be empirically scored; (iii) the red-flag baseline substitutes five proxy flags rather than the exact bidder-price rules; (iv) no independently adjudicated labels; (v) original cap of 100k lots waived; (vi) all extended calculations reside in this reproducible notebook rather than 12 standalone scripts; (vii) primary synthetic patterns favor Graph IF and are not independent labels. Do not claim that a generated PDF proves the model detects real-world procurement violations.

Review packet and exact SHA-256/commit log require additional author-controlled steps. The repository scaffold accompanying this notebook is a starting point, not a published GitHub repository.

## Part XV — Handoff figures and blinded review workbook

The following files are deliberately **restricted**: raw identifiers, bidders, amounts and offer prices are not suitable for an unaudited public repository. Blinded reviewer data omit ranking, scores, method flags and graph fields; the secret key must remain with study coordinators. Reviewers should inspect evidence independently. The actual review labels and source URLs are not invented. The source does not guarantee public URLs for all lots.

In [ ]:
# Reviewer case selection: 40 each from seed-42 top 5% + 100 random, deduplicated.
review_rng=np.random.default_rng(20261013)
rank_views={'red_flag':v3_rankings.red_flag_score.to_numpy(dtype=float),
            'tabular':v3_scores_t[42],'graph':v3_scores_g[42]}
chosen_by=defaultdict(list)
for method,sc in rank_views.items():
    candidate=np.array(sorted(topids(test_ids,sc,.05)))
    for identifier in review_rng.choice(candidate,size=min(40,len(candidate)),replace=False):
        chosen_by[str(identifier)].append(method)
for identifier in review_rng.choice(test_ids,size=min(100,len(test_ids)),replace=False):
    chosen_by[str(identifier)].append('random')
review_lots=list(chosen_by)
review_rng.shuffle(review_lots)
review_map=f.loc[test,['lot_id','buy_id','cutoff_utc','amount_kzt','trade_method','category2','start_utc']].copy()
review_map['lot_id']=review_map.lot_id.astype(str)
review_map=review_map.set_index('lot_id')
# Show observed source applicant IDs and prices only in restricted workbook.
observed_prices=offers[['trd_lots_id','sys_subjects_id','price']].copy()
observed_prices['lot_id']=canonical_id(observed_prices.trd_lots_id)
observed_prices['supplier_id']=canonical_id(observed_prices.sys_subjects_id)
observed_prices['bid_price']=pd.to_numeric(observed_prices.price,errors='coerce')
observed_prices=observed_prices.loc[observed_prices.lot_id.isin(review_lots)]
price_map=observed_prices.groupby('lot_id',sort=False).apply(
    lambda x:'; '.join(f'{r.supplier_id}: {r.bid_price:.2f}' if pd.notna(r.bid_price) else str(r.supplier_id)
                       for r in x.itertuples()),include_groups=False).to_dict()
case_records=[];key_records=[]
for i,lot in enumerate(review_lots):
    record=review_map.loc[lot]
    alias='CASE_'+format(i+1,'04d')
    case_records.append({'case_id':alias,'buy_id':str(record.buy_id),
        'method':str(record.trade_method),'category':str(record.category2),
        'amount_kzt':record.amount_kzt,'publication_proxy':str(record.start_utc),
        'deadline_proxy':str(record.cutoff_utc),
        'applicants_and_bid_prices_RESTRICTED':price_map.get(lot,''),
        'source_url':'',
        'reviewer1_review_worthy':'','reviewer1_reason':'',
        'reviewer2_review_worthy':'','reviewer2_reason':''})
    row={'case_id':alias,'lot_id':lot,'selection_strata':';'.join(chosen_by[lot])}
    for method,sc in rank_views.items():
        row[method+'_rank']=int(stable_rank(test_ids,sc)[np.where(test_ids==lot)[0][0]])
    key_records.append(row)
blind=pd.DataFrame(case_records);secret=pd.DataFrame(key_records)
assert not any('score' in c or 'rank' in c or 'selection' in c for c in blind.columns)
blind.to_excel(V3_DIR/'review_sheet_RESTRICTED.xlsx',index=False)
secret.to_csv(V3_DIR/'review_key_RESTRICTED.csv',index=False)
print('Blind cases:',len(blind),'— preserve secret key separately; never upload either to public GitHub.')

# Data-flow chart reflects the accounting already computed in the previous notebook parts.
flow_counts=[('Raw exported lots',len(lots)),
    ('Eligible identifiers/deadlines',int((cohort.audit_reason=='eligible_for_version_review').sum())),
    ('Pilot Oct 2025–Sep 2026',len(pilot)),
    ('Train',int(train.sum())),('Validation',int(val.sum())),('Test',int(test.sum()))]
fig,ax=plt.subplots(figsize=(9,5))
for i,(name,value) in enumerate(flow_counts):
    ax.text(.5,1-(i+.5)/len(flow_counts),f'{name}: {value:,}',ha='center',va='center',
        transform=ax.transAxes,bbox={'boxstyle':'round,pad=.4','fc':'white','ec':'#777777'})
    if i<len(flow_counts)-1:
        ax.annotate('',xy=(.5,1-(i+1)/len(flow_counts)+.04),
                    xytext=(.5,1-(i+1)/len(flow_counts)+.08),
                    xycoords='axes fraction',textcoords='axes fraction',
                    arrowprops={'arrowstyle':'->','lw':1.3})
ax.set_axis_off();fig.savefig(V3_DIR/'fig_data_flow.pdf',bbox_inches='tight');plt.show()

# Score distribution split by observed bidder graph coverage, not 'verified public history'.
fig,axes=plt.subplots(1,2,figsize=(11,4),constrained_layout=True)
cover=f.loc[test,'v3_n_observed_bidders'].to_numpy()>0
for ax,(model,sc) in zip(axes,[('Tabular IF',original_t),('Graph IF',original_g)]):
    for label,mask in [('Observed bidders',cover),('No observed bidders',~cover)]:
        if mask.any():ax.hist(sc[mask],bins=35,alpha=.5,density=True,label=label)
    ax.set(xlabel='Anomaly score (higher = more anomalous)',ylabel='Density')
    ax.legend(frameon=False)
fig.savefig(V3_DIR/'fig_score_dist.pdf',bbox_inches='tight');plt.show()

# Disagreement cases, identifiers restricted, NO invented causal explanation.
err=v3_rankings[['lot_id','red_flag_rank','tabular_rank_seed_42','graph_rank_seed_42',
                 'red_flag_score','tabular_score_seed_42','graph_score_seed_42']].copy()
err['max_rank_gap']=(err[['red_flag_rank','tabular_rank_seed_42','graph_rank_seed_42']].max(axis=1)-
                     err[['red_flag_rank','tabular_rank_seed_42','graph_rank_seed_42']].min(axis=1))
a=err.nsmallest(20,'graph_rank_seed_42').assign(reason='graph_rank_top20_explain_with_source_evidence')
b=err.nlargest(20,'max_rank_gap').assign(reason='cross_model_disagreement_unexplained')
pd.concat([a,b],ignore_index=True).drop_duplicates('lot_id').to_csv(V3_DIR/'tab_errors_RESTRICTED.csv',index=False)
print('Additional handoff formats saved in',V3_DIR)
